> ## SOLUTION / ANSWER KEY &mdash; Lab 5.12
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-12-capstone-mini-agent.ipynb`](../lab-12-capstone-mini-agent.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 5.12 &mdash; Capstone: A Mini Autonomous Agent

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 45 min &nbsp;|&nbsp; **Day 3 &middot; Module 5 &mdash; What is Agentic AI?**

### What you'll do
- Assemble the whole module: tools + memory + guardrails + the loop
- Run the agent over a SUITE of different tasks
- Score tasks solved / total and confirm guardrails hold

> **How this lab works (experiential flow):** read the **Concept**, run the **Demo** to see it work, then complete **Your Turn** by replacing every `___` placeholder. Run the **grader** cell at the end &mdash; it prints `[PASS]` / `[FAIL]` / `[TODO]` and a final `Score`. Aim for a full score.

**Reference:** [Module 5 slides &mdash; A day in the life of an agent](../../presentation/day3-module5-what-is-agentic-ai.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 5 labs](./index.html)

In [ ]:
# Setup -- run me first
import os
WORK = "/tmp/biaa-lab-05-12"
os.makedirs(WORK, exist_ok=True)
print("Working dir:", WORK)

## Concept
Capstone: a **mini autonomous agent** that brings together everything &mdash; a tool registry, a
reason/act/observe **loop**, **memory**, and a **guardrail** (only allowed tools) &mdash; and runs
it over a **suite** of tasks (a two-step lookup-and-compute, a fact lookup, and an arithmetic task).
The score reflects **tasks solved / total**. The optional cell swaps the rule-based policy for a
**real LangChain agent** &mdash; the bridge to Module 6.

In [ ]:
import ast, operator
# safe arithmetic: walk a parsed AST of numbers + (+ - * / ** unary-minus) -- never bare eval()
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.USub: operator.neg, ast.Pow: operator.pow}
def safe_calc(expr):
    def ev(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp):
            return _OPS[type(node.op)](ev(node.left), ev(node.right))
        if isinstance(node, ast.UnaryOp):
            return _OPS[type(node.op)](ev(node.operand))
        raise ValueError("unsupported expression")
    return ev(ast.parse(expr, mode="eval").body)
KB = {"population of metropolis": "120000", "capital of france": "Paris"}
TOOLS = {"calculator": safe_calc, "lookup": lambda k: KB.get(k.lower().strip(), "unknown")}
ALLOWED = set(TOOLS)
print("tools:", list(TOOLS), "| allow-list:", ALLOWED)

## Your Turn
Complete the policy routing, the guardrail check, and the suite evaluation. `run_agent` ties it together.

In [ ]:
def policy(goal, memory):
    seen = [s["action"] for s in memory]
    text = goal.lower()
    if "population" in text and "twice" in text:
        if "lookup" not in seen:
            return ("lookup", "population of metropolis")
        if "calculator" not in seen:
            pop = int([s["observation"] for s in memory if s["action"] == "lookup"][0])
            return ("calculator", str(pop) + "*2")
        return ("final", str([s["observation"] for s in memory if s["action"] == "calculator"][0]))
    if "capital" in text:
        if "lookup" not in seen:
            return ("lookup", "capital of france")
        return ("final", [s["observation"] for s in memory if s["action"] == "lookup"][0])
    if "compute" in text:
        if "calculator" not in seen:
            return ("calculator", goal.split("compute")[-1].strip())
        return ("final", str([s["observation"] for s in memory if s["action"] == "calculator"][0]))
    return ("final", "I cannot solve this")

def run_agent(goal, max_steps=6):
    memory = []
    for _ in range(max_steps):
        action, arg = policy(goal, memory)
        if action == 'final':
            return str(arg)
        if action not in ALLOWED:
            return 'blocked'
        obs = TOOLS[action](arg)
        memory.append({'action': action, 'observation': obs})
    return 'max_steps'

SUITE = [
    {"goal": "twice the population of metropolis", "answer": "240000"},
    {"goal": "what is the capital of france", "answer": "Paris"},
    {"goal": "compute 15*3", "answer": "45"},
]
def evaluate():
    solved = sum(1 for t in SUITE if run_agent(t["goal"]) == t["answer"])
    return solved, len(SUITE)

try:
    for t in SUITE:
        print(t['goal'], '->', run_agent(t['goal']))
    print('solved:', evaluate())
except Exception as e:
    print('Fill the blanks, then re-run.', type(e).__name__)

In [ ]:
# === Auto-grader: run after filling the blanks above ===
_results = []
def _rec(label, status, extra=""):
    _results.append(status); print(f"[{status}] {label}" + (f" -- {extra}" if extra else ""))
def expect(label, got, want):
    if got == "___" or got is None: _rec(label, "TODO")
    elif got == want: _rec(label, "PASS")
    else: _rec(label, "FAIL", f"got {got!r}")
def expect_true(label, fn):
    try: _rec(label, "PASS" if fn() else "FAIL")
    except Exception as e: _rec(label, "TODO", type(e).__name__)

expect_true("solves the two-step population task", lambda: run_agent("twice the population of metropolis") == "240000")
expect_true("solves the capital lookup task", lambda: run_agent("what is the capital of france") == "Paris")
expect_true("solves the arithmetic task", lambda: run_agent("compute 15*3") == "45")
expect_true("solves the whole suite (3/3)", lambda: evaluate() == (3, 3))
expect_true("handles an unsolvable goal without crashing", lambda: run_agent("xyzzy nonsense") == "I cannot solve this")
expect_true("never exceeds the step cap", lambda: run_agent("twice the population of metropolis") != "max_steps")

_p = _results.count("PASS")
print(f"\nScore: {_p}/{len(_results)}")
print("All checks passed -- lab complete!" if _p == len(_results) else "Keep going: fill the blanks marked ___ and re-run.")

## Optional &mdash; swap the rule-based policy for a REAL LLM (not graded)
Swap the rule-based policy for a REAL LangChain agent (Ollama llama3.2:1b, or Groq) -- the bridge to Module 6. Safe to skip &mdash; it needs a local **Ollama** (`pip install langchain-ollama`, then
`ollama run llama3.2:1b`) or a **Groq** key (`pip install langchain-groq`, `GROQ_API_KEY`). If
neither is present the cell prints a friendly note and moves on. **The graded steps above run on a
deterministic rule-based policy, so the lab always verifies offline.**

In [ ]:
try:
    from langchain_ollama import ChatOllama
    llm = ChatOllama(model="llama3.2:1b")
    print(llm.invoke("In one sentence, what is an AI agent?").content)
    print("That reply came from a REAL local LLM -- in Module 6 you bind it to tools with LangChain")
    print("and let IT drive the loop, instead of the rule-based policy above.")
except Exception as e:
    print("No local LLM available -- skipping (install langchain-ollama + `ollama run llama3.2:1b`,", type(e).__name__)
    print("or use Groq: `from langchain_groq import ChatGroq` with GROQ_API_KEY).")
    print("The rule-based mini-agent above already solved the whole suite offline.")
    print("Next: Module 6 (Agent Frameworks) and your Day-3 labs -- a simple LangChain agent, then")
    print("connecting agents to external APIs (Google Search / Wolfram Alpha).")

---
### Key takeaway
You built a guardrailed mini-agent that solves a suite of tasks with tools, memory and a loop. That IS agentic AI &mdash; next, Module 6 hands the loop to a real LLM with LangChain.

[&#8592; All Module 5 labs](./index.html) &nbsp;&middot;&nbsp; [Module 5 slides](../../presentation/day3-module5-what-is-agentic-ai.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>